In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!ls "/content/drive/MyDrive/Colab Notebooks/"

 10685642.zip
 aplication_ai_mlp.ipynb
 app_ai_scaler.pkl
 app_ensemble_threshold.npy
'application_dataset (1).csv.gz'
 application_dataset.csv.gz
 application_gatekeeper.ipynb
 application_gatekeeper_streaming.pkl
 application_logs_kafka.ipynb
 application_test_logs_malicious_heavy.csv
 Application_valid_layer.ipynb
 app_test_logs.csv
'ca (1).pem'
 ca_b.pem
 ca.pem
 client.truststore.jks
'Copy of NF-ToN-IoT-v3.csv'
 cryptography.ipynb
 csic_final.csv
'cybersecurity_threat_detection_logs (1).csv'
 cybersecurity_threat_detection_logs.csv
 dataset_20_percent.csv
 deep_ai_model.h5
 deep_ai_scaler.pkl
 df_refined.csv
 ensemble_threshold.npy
 feature_app_columns.pkl
 feature_columns.pkl
 files_split.ipynb
 forensic_audit.jsonl
 forensic_explanation.ipynb
 forensic_reports.json
 gatekeeper_network.pkl
 kept_logs_for_training.csv
 lgbm_model.pkl
 log_file_1972.csv
 log_files_iot.zip
 meta_model.pkl
 mlp_application_model.h5
 mlp_feature_columns.pkl
 mlp_label_encoders.pkl
 mlp_model.keras
 ml

In [4]:
#Detection
import pandas as pd
import joblib

FILE_PATH="/content/drive/MyDrive/Colab Notebooks/"

# ===============================
# LOAD MODEL
# ===============================
package=joblib.load(FILE_PATH+"windows_threat_model.pkl")

model=package["model"]
features=package["features"]

# ===============================
# LOAD FULL DATASET
# ===============================
def feature_engineering(df):

    df.columns=df.columns.str.strip().str.replace(" ","_").str.lower()

    df["date_and_time"]=pd.to_datetime(
        df["date_and_time"],
        format="%d.%m.%Y. %H:%M:%S",
        errors="coerce"
    )

    df=df.sort_values("date_and_time")

    df["hour"]=df["date_and_time"].dt.hour

    df["night_activity"]=((df["hour"]<6)|(df["hour"]>22)).astype(int)

    df["event_freq"]=df.groupby("event_id").cumcount()

    df["burst_count"]=df.groupby("event_id").cumcount().clip(upper=20)

    return df

df=pd.read_csv(FILE_PATH+"uploaded_logs.csv")
#df=pd.read_csv(FILE_PATH+"windows_labeled_logs_dataset.csv")

df = feature_engineering(df)
print("Logs loaded:",len(df))

#if "label" in df.columns:
 #   df=df.drop(columns=["label"])

# ===============================
# PREDICTION
# ===============================

X=df[features]

probs = model.predict_proba(X)[:,1]

df["risk_score"]=0

df.loc[probs>0.50,"risk_score"]=1
df.loc[probs>0.97,"risk_score"]=2

print("\nRisk distribution")
print(df["risk_score"].value_counts())

# ===============================
# SAVE OUTPUT
# ===============================

df.to_csv(FILE_PATH+"windows_ai_output.csv",index=False)


print("AI detection completed")

Logs loaded: 3522

Risk distribution
risk_score
0    3022
1     261
2     239
Name: count, dtype: int64
AI detection completed


In [ ]:
# !ls "/content/drive/MyDrive/Colab Notebooks/"

In [ ]:
# import pandas as pd
# import joblib

# from sklearn.model_selection import train_test_split
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import classification_report
# from sklearn.metrics import confusion_matrix
# from sklearn.metrics import accuracy_score

# FILE_PATH="/content/drive/MyDrive/Colab Notebooks/"

# df=pd.read_csv(FILE_PATH+"windows_processed_logs.csv")

# print("Logs loaded:",len(df))

# # ==============================
# # FEATURES
# # ==============================

# features=[

# "Event_ID",
# "hour_sin",
# "hour_cos",
# "night_activity",
# "event_freq",
# "event_freq_log",
# "burst_count",
# "is_privileged_event",
# "event_rarity"

# ]

# X=df[features]

# y=df["label"]

# # ==============================
# # TRAIN TEST SPLIT
# # ==============================

# X_train,X_test,y_train,y_test=train_test_split(

# X,
# y,
# test_size=0.2,
# random_state=42,
# stratify=y

# )

# # ==============================
# # STRICT RANDOM FOREST
# # ==============================

# model=RandomForestClassifier(

# n_estimators=600,
# max_depth=20,
# min_samples_split=2,
# min_samples_leaf=1,
# random_state=42,
# n_jobs=-1

# )

# model.fit(X_train,y_train)

# # ==============================
# # PREDICTIONS
# # ==============================

# y_pred=model.predict(X_test)

# # ==============================
# # EVALUATION
# # ==============================

# print("\nAccuracy:",accuracy_score(y_test,y_pred))

# print("\nClassification Report")
# print(classification_report(y_test,y_pred))

# print("\nConfusion Matrix")
# print(confusion_matrix(y_test,y_pred))

# # ==============================
# # SAVE MODEL
# # ==============================
# package = {
#     "model": model,
#     "features": features
# }

# joblib.dump(package, FILE_PATH+"windows_threat_detection_model.pkl")

# print("\nModel saved successfully")

Logs loaded: 84065

Accuracy: 0.9550347945042527

Classification Report
              precision    recall  f1-score   support

           0       0.96      0.98      0.97      5098
           1       0.92      0.88      0.90      2245
           2       0.96      0.96      0.96      5597
           3       0.96      0.97      0.96      3873

    accuracy                           0.96     16813
   macro avg       0.95      0.95      0.95     16813
weighted avg       0.95      0.96      0.95     16813


Confusion Matrix
[[4993   85   20    0]
 [ 162 1974   97   12]
 [  32   72 5349  144]
 [   0   10  122 3741]]

Model saved successfully


In [ ]:
# from google.colab import drive
# import os

# # Unmount Google Drive
# drive.flush_and_unmount()
# print("Drive unmounted")

# # Terminate runtime session
# os.kill(os.getpid(), 9)